In [16]:
%pip install pandas numpy


Note: you may need to restart the kernel to use updated packages.


In [9]:
import pandas as pd
import numpy as np

# ==========================================
# 1. BUILD DATA PACKET (BYPASSING DICT STRIP)
# ==========================================
# We create individual list elements so the engine preserves the code structure
prop_ids = ["PROP-001", "PROP-002", "PROP-003", "PROP-004", "PROP-005"]
locations = ["Airport Residential, Accra", "Cape Coast (Near UCC)", "Kumasi Asokwa", "East Legon, Accra", "Elmina Coastline"]

# Populate numeric data arrays directly
values = [5000000, 1200000, 2500000, 4800000, 950000]
rentals = [350000, 84000, 180000, 320000, 76000]
risks = [1, 2, 1, 2, 5]
coords = ["5.606, -0.168", "5.114, -1.282", "6.674, -1.602", "5.632, -0.151", "5.085, -1.348"]

# Zip lists directly into a clean Pandas DataFrame
raw_listings = pd.DataFrame(
    list(zip(prop_ids, locations, values, rentals, risks, coords)),
    columns=["Property_ID", "Location", "Market_Value_GHS", "Annual_Rental_Income_GHS", "Zoning_Risk_Score", "GIS_Lat_Long"]
)

# ==========================================
# 2. ANALYTICS & RISK ASSESSMENT PIPELINE
# ==========================================
def process_rwa_pipeline(df):
    """Processes financials, applies geographic risk adjustments, and models tokenization."""
    # Calculate Gross Rental Yield
    df["Gross_Yield_Pct"] = (df["Annual_Rental_Income_GHS"] / df["Market_Value_GHS"]) * 100
    
    # Apply Risk-Adjusted Valuation (Geography-Driven Metric)
    def calculate_haircut(row):
        if row["Zoning_Risk_Score"] >= 4:
            return row["Market_Value_GHS"] * 0.80 # 20% value penalty for environmental/legal exposure
        return row["Market_Value_GHS"]

    df["Risk_Adjusted_Value_GHS"] = df.apply(calculate_haircut, axis=1)
    
    # Tokenization Modeling (100 GHS per token baseline)
    TOKEN_PRICE_GHS = 100.00
    df["Total_Tokens_Issued"] = (df["Risk_Adjusted_Value_GHS"] / TOKEN_PRICE_GHS).astype(int)
    
    # Add Tokenization Status
    df["RWA_Compliance_Status"] = np.where(df["Zoning_Risk_Score"] >= 4, "REJECTED (High Risk)", "APPROVED FOR ON-CHAIN LISTING")
    
    return df

# Execute and process portfolio metrics
processed_rwa = process_rwa_pipeline(raw_listings)

# Save clean CSV output into your active environment folder
processed_rwa.to_csv("rwa_tokenized_portfolio.csv", index=False)

# Render the formatted asset verification matrix
processed_rwa[["Property_ID", "Location", "Market_Value_GHS", "Zoning_Risk_Score", "Total_Tokens_Issued", "RWA_Compliance_Status"]]


,Property_ID,Location,Market_Value_GHS,Zoning_Risk_Score,Total_Tokens_Issued,RWA_Compliance_Status
0,PROP-001,"Airport Residential, Accra",5000000,1,50000,APPROVED FOR ON-CHAIN LISTING
1,PROP-002,Cape Coast (Near UCC),1200000,2,12000,APPROVED FOR ON-CHAIN LISTING
2,PROP-003,Kumasi Asokwa,2500000,1,25000,APPROVED FOR ON-CHAIN LISTING
3,PROP-004,"East Legon, Accra",4800000,2,48000,APPROVED FOR ON-CHAIN LISTING
4,PROP-005,Elmina Coastline,950000,5,7600,REJECTED (High Risk)
